# Fan 2023 · handwriting CORP T5

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `CORP_data_release.zip` from [Dryad](https://doi.org/10.5061/dryad.hqbzkh1p6). Create `studies/20231201_fan/data/raw/dryad_hqbzkh1p6/` and save the download as `studies/20231201_fan/data/raw/dryad_hqbzkh1p6/dryad_hqbzkh1p6__CORP_data_release.zip`.

### 2. Extract

Extract the archive into `studies/20231201_fan/data/extracted/dryad_hqbzkh1p6/`.

```text
python -m zipfile -e "studies/20231201_fan/data/raw/dryad_hqbzkh1p6/dryad_hqbzkh1p6__CORP_data_release.zip" "studies/20231201_fan/data/extracted/dryad_hqbzkh1p6"
```

After extraction, check for `studies/20231201_fan/data/extracted/dryad_hqbzkh1p6/CORP_data_release/`.

### 3. Prepare the model input

The replay reads the released TFRecords directly. These are the training and replay input directories.

- Seed training `studies/20231201_fan/data/extracted/dryad_hqbzkh1p6/CORP_data_release/seed_model_training_data/tfrecords/`
- Online replay `studies/20231201_fan/data/extracted/dryad_hqbzkh1p6/CORP_data_release/online_evaluation_data/recalibration/tfrecords/`

The paired WER replay also reads reference trials from `studies/20231201_fan/data/raw/dryad_hqbzkh1p6/dryad_hqbzkh1p6__CORP_data_release.zip`. Set up the upstream CORP code before running the replay.

```text
python shared/tools/setup_upstream.py --study 20231201_fan
```

### 4. Export the common session file

Write one follow-up session to the session format shared by every study in this repository. The file stores every trial of both arms with the features as released, and records the TFRecord shard, the example position, and the row of the session's sentence table. This example exports September 29, 2022.

```text
python studies/20231201_fan/code/export_sessions.py --session t5.2022.09.29
```

The exporter reads the TFRecords from the extracted tree. It reads the sentence and block tables from the `mat/` members inside the ZIP, not from the extracted tree, so the ZIP from step 1 stays required.

**Output** `studies/20231201_fan/data/processed/t5/dryad_hqbzkh1p6_2022-09-29.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its publisher file with one command.

```text
python shared/tools/validate_session.py studies/20231201_fan/data/processed/t5/dryad_hqbzkh1p6_2022-09-29.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 60/60 trials re-read and identical`.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20231201_fan/data/processed/t5/dryad_hqbzkh1p6_2022-09-29.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = next(t for t in session.trials if t.attrs["source_arm"] == "recalibration")
        print(trial.trial_id, "·", trial.target("sentence"))
        array = trial.neural("input_features")
        print(f"  neural/input_features: {array.shape} {array.dtype}")
        chars = session.label_sets["chars"]
        print("  characters:", "".join(chars[i] for i in trial.target("char_class_ids")[:12]), "...")
        print(f"  sentence table row {trial.attrs['mat_index']}, block {trial.attrs['source_block']}")
else:
    print("Session file not found. Run the export command above to create it.")

t5.2022.09.29: 60 trials
t5.2022.09.29/recalibration/train/0000 · what targets still remain to be hit?
  neural/input_features: (1799, 192) float32
  characters: what>targets ...
  sentence table row 0, block 3


## Read the archived results

The final six trials of each session were withheld from CORP adaptation. Figure 3b compares greedy CER to WER from a character 5-gram beam. This replay covers an alternate decoder and has a separate full-pipeline status in Figure 2f.

This walkthrough uses the checked-in archive and takes no data download.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
from build_session_cer_vs_wer import build_fan23
source = benchmark["figure3_cer_sources"]["Fan 2023 (T5)"]
trial = read_rows(source["archive"])[0]
print(trial["reference"], "→", trial["beam_text"])
print(f"Trial beam word errors: {trial['beam_word_errors']}/{trial['reference_words']}")
result = build_fan23()
print("First replay day:", result["sessions"][0])
print(f"{len(result['sessions'])} sessions, {sum(d['n'] for d in result['sessions'])} withheld trials; pooled {result['pooled']}")

but this year prospects may be better. → but this year prospecs may be better.
Trial beam word errors: 1/7
First replay day: {'session': 't5.2022.09.29', 'n': 6, 'cer': 4.94, 'wer': 20.83, 'char_counts': 263, 'word_counts': 48, 'char_errors': 13, 'word_errors': 10}
10 sessions, 60 withheld trials; pooled {'cer': 12.29, 'wer': 41.8}


## Continue the full reproduction

For paired WER, pass `--study studies/20231201_fan` to `evaluate.py`.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`export_sessions.py`](code/export_sessions.py), [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py). Run a command with `--help` to see its arguments.